# Generic tabular fairness pipeline — modular v18.0 — corrected 1000-bootstrap custom intersectional mitigation

This corrected version aligns the raw runtime/input configuration and the effective fairness configuration. The final mitigation configuration uses `bootstrap_reps=1000`, `mitigation_scope="custom_plus_pairs"`, and all requested 2-, 3-, and 4-way intersectional groups among the configured custom sensitive attributes when subgroup support is sufficient.

The configuration is intentionally heavier than the previous runtime-safe smoke version. A commented smoke-test override remains available in the analyst configuration cell for installation or parsing checks.

## 0. Dependency installation attempt

This must be the first executable step. The cell first installs the core scientific stack, then attempts to install Fairlearn, AIF360, and imbalanced-learn. Pip version notices are disabled to keep the notebook output clean. If the installation or import fails, the notebook records the failure and continues with built-in support operations.

In [ ]:
# =============================================================================
# FIRST EXECUTABLE CELL: DEPENDENCY INSTALLATION ATTEMPT
# =============================================================================
import os
import sys
import subprocess
import warnings
from pathlib import Path

# Suppress the pip version notice while still showing real installation failures.
os.environ.setdefault("PIP_DISABLE_PIP_VERSION_CHECK", "1")
os.environ.setdefault("PYTHONWARNINGS", "ignore::FutureWarning")
warnings.filterwarnings("ignore", message=r".*Setting an item of incompatible dtype.*", category=FutureWarning, module=r"fairlearn\.postprocessing.*")
warnings.filterwarnings("ignore", message=r".*has dtype incompatible with.*", category=FutureWarning, module=r"fairlearn\.postprocessing.*")

STATUS_FILE = Path("/tmp/tfm_fairness_install_status.txt")
os.environ["TFM_FAIRNESS_INSTALL_STATUS"] = str(STATUS_FILE)

print("[Setup] Python:", sys.version.split()[0])
print("[Setup] Working directory:", Path.cwd())

install_messages = []

def _pip_install(requirements_file: str, label: str) -> bool:
    req = Path(requirements_file)
    if not req.exists():
        msg = f"[Setup] {label}: {requirements_file} not found; continuing with installed packages."
        print(msg)
        install_messages.append(msg)
        return False
    print(f"[Setup] Installing {label} from {requirements_file}...")
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check",
        "--no-input",
        "-q",
        "-r", str(req),
    ]
    try:
        proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=False)
        if proc.returncode == 0:
            msg = f"[Setup] {label}: installation attempt completed."
            print(msg)
            install_messages.append(msg)
            return True
        msg = f"[Setup] {label}: installation attempt failed with code {proc.returncode}. Support operations will be used where needed."
        print(msg)
        stderr_clean = "\n".join(line for line in proc.stderr.splitlines() if not line.strip().startswith("[notice]"))
        if stderr_clean.strip():
            print(stderr_clean.strip()[-2000:])
        install_messages.append(msg)
        return False
    except Exception as exc:
        msg = f"[Setup] {label}: installation attempt failed: {repr(exc)}"
        print(msg)
        install_messages.append(msg)
        return False

_pip_install("requirements_core.txt", "core scientific packages")
_pip_install("requirements_fairness.txt", "external fairness libraries: fairlearn + aif360")

if not Path("requirements_core.txt").exists() and not Path("requirements_fairness.txt").exists():
    _pip_install("requirements.txt", "full requirements")

for module_name in ("fairlearn", "aif360", "imblearn"):
    try:
        __import__(module_name)
        msg = f"[Setup] {module_name}: import OK."
    except Exception as exc:
        msg = f"[Setup] {module_name}: import unavailable after install attempt; support operations will be used where needed. {repr(exc)}"
    print(msg)
    install_messages.append(msg)

STATUS_FILE.write_text("\n".join(install_messages), encoding="utf-8")
print(f"[Setup] Installation status recorded at: {STATUS_FILE}")


## 1. Minimal dataset input for raw EDA

At this point, only the CSV location and runtime/output preferences are needed. The target and fairness variables are configured after the raw EDA.


In [ ]:
# =============================================================================
# MINIMAL INPUT CONFIGURATION FOR RAW CSV EDA + FINAL THESIS MITIGATION SETTINGS
# =============================================================================
# Edit csv_path here if your dataset is not in the current folder.
#
# Correction implemented in v18:
# - The raw-EDA configuration and final fairness configuration now carry the same
#   bootstrap and mitigation-scope values, so tables 00 and 06 do not contradict
#   each other.
# - Final mitigation uses 1000 bootstrap repetitions.
# - Final mitigation keeps mitigation_scope="custom_plus_pairs" but explicitly
#   includes 2-, 3-, and 4-way intersectional groups among the custom attributes
#   when subgroup support is sufficient.

FINAL_MITIGATION_SETTINGS = {
    "fairness_audit_scope": "single_plus_intersectional",
    "mitigation_scope": "custom_plus_pairs",
    "mitigation_custom_attributes": (
        "gender_label", "race_label", "homo_label", "drugs_label",
        "ever_married", "Residence_type", "work_type",
    ),
    # Generates all pairs, triples and four-way groups among the custom attributes
    # that are present in the dataset. For ACTG175, this normally means the
    # available variables gender/race/homo/drugs and their 2-, 3-, and 4-way groups.
    "mitigation_custom_intersections": "all_intersections_among_custom_attributes",
    "mitigation_intersectional_orders": (2, 3, 4),
    "mitigation_max_intersectional_groups": 80,
    "mitigation_max_intersectional_attributes": 200,
    "mitigation_max_jobs": 20000,
    "mitigation_top_k_attributes": 7,
    "mitigation_max_accuracy_drop": 0.05,
    "min_group_size_for_mitigation": 20,
    "bootstrap_reps": 1000,
    "significance_alpha": 0.05,
}

DATASET_INPUT = {
    "runtime": {
        "environment": "auto",          # "auto", "local", "colab", or "kaggle"
        "execution_preset": "full",     # Corrected final run. Use "smoke" only for a fast test run.
        "output_base_dir": None,         # None -> runtime-specific default/current folder
        "n_jobs": None,
        "use_gpu": "auto",              # detects and reports GPU availability in every environment
        "xgboost_device_policy": "safe_cpu",  # safe_cpu avoids XGBoost CUDA/CPU prediction mismatch in sklearn pipelines
        "show_mitigation_progress": True,
        "mitigation_progress_detail": "standard",  # "standard" or "verbose"
        "require_fairness_libraries_first": True,
    },
    "data": {
        "dataset_mode": "auto",         # auto detects the included clinical example; generic_tabular for a custom CSV
        "dataset_name": "clinical_example",
        "csv_path": "example_clinical_dataset.csv",
        "csv_candidates": ("example_clinical_dataset.csv", "dataset.csv", "data.csv"),
        "csv_read_kwargs": {},
    },
    "fairness": FINAL_MITIGATION_SETTINGS,
}


## 2. Package import and raw CSV EDA

This stage extracts the CSV schema, detects variable types, checks missingness and duplicates, and displays visual EDA before the fairness pipeline is configured.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from fairness_pipeline.workflow import (
    initialise_eda,
    stage_1_raw_dataset_eda,
    initialise_pipeline,
    stage_2_configured_dataset_preparation,
    stage_3_train_models,
    stage_4_baseline_visual_evaluation,
    stage_5_fairness_audit,
    stage_6_mitigation_and_significance,
    stage_7_finalise,
    stage_8_export_results_zip,
)

eda_ctx = initialise_eda(DATASET_INPUT)
eda_ctx = stage_1_raw_dataset_eda(eda_ctx)


## 3. Analyst configuration checkpoint

Review the raw EDA outputs above before editing this cell. Keep `feature_cols=None` for automatic feature selection, or provide an explicit list if the analysis plan requires it. For a generic CSV, set `dataset_mode='generic_tabular'`, `target_col`, `positive_target_value`, and `sensitive_attrs`.


In [ ]:
# =============================================================================
# USER CONFIGURATION CELL FOR THE FAIRNESS ANALYSIS
# =============================================================================
# This is the main analyst-editable cell. The rest of the notebook reads USER_CONFIG.
# Review the raw EDA first, then modify this configuration according to the dataset.
#
# v18 correction: the final fairness settings below are intentionally the same as
# FINAL_MITIGATION_SETTINGS in the raw-EDA input cell. This prevents the previous
# mismatch where the raw runtime table and the effective fairness table reported
# different bootstrap and mitigation-scope values.

USER_CONFIG = {
    "runtime": {
        "environment": DATASET_INPUT["runtime"].get("environment", "auto"),
        "execution_preset": DATASET_INPUT["runtime"].get("execution_preset", "full"),
        "output_base_dir": DATASET_INPUT["runtime"].get("output_base_dir"),
        "n_jobs": DATASET_INPUT["runtime"].get("n_jobs"),
        "use_gpu": DATASET_INPUT["runtime"].get("use_gpu", "auto"),
        "xgboost_device_policy": DATASET_INPUT["runtime"].get("xgboost_device_policy", "safe_cpu"),
        "show_mitigation_progress": DATASET_INPUT["runtime"].get("show_mitigation_progress", True),
        "mitigation_progress_detail": DATASET_INPUT["runtime"].get("mitigation_progress_detail", "standard"),
    },
    "data": {
        # For the included clinical example, leave dataset_mode="auto".
        # For another CSV, set dataset_mode="generic_tabular" and define target_col below.
        "dataset_mode": DATASET_INPUT["data"].get("dataset_mode", "auto"),
        "dataset_name": DATASET_INPUT["data"].get("dataset_name", "clinical_example"),
        "scenario": "baseline",
        "csv_path": DATASET_INPUT["data"].get("csv_path"),
        "csv_candidates": DATASET_INPUT["data"].get("csv_candidates"),
        "csv_read_kwargs": DATASET_INPUT["data"].get("csv_read_kwargs", {}),
        "target_col": None,                  # e.g. "outcome" for a generic CSV
        "positive_target_value": 1,
        "sensitive_attrs": "auto",           # auto includes common sensitive attributes when present
        "exclude_cols": (),                  # e.g. ("patient_id", "date", "leakage_variable")
        "feature_cols": None,                # None -> automatic feature selection
        "include_sensitive_as_features": False,
    },
    "models": {
        "enabled_models": None,              # None -> preset default; full preset trains all available default models
        "scoring_metric": "roc_auc",
        "cv_folds": None,
        "max_grid_per_model": None,
    },
    "fairness": dict(FINAL_MITIGATION_SETTINGS),
    "reporting": {
        "display_figures": True,
        "show_model_training_explanations": True,
        "max_table_rows_display": 120,
        "show_mitigation_progress": True,
        "mitigation_progress_detail": "standard",
    },
}

# Fast smoke-test alternative: useful only to check installation and dataset parsing.
# This changes runtime only; it deliberately leaves the final mitigation settings visible.
# USER_CONFIG["runtime"]["execution_preset"] = "smoke"
# USER_CONFIG["fairness"].update({
#     "bootstrap_reps": 20,
#     "mitigation_max_jobs": 40,
#     "mitigation_max_intersectional_attributes": 25,
#     "mitigation_methods": ("postprocess_group_threshold_equalized_odds",),
# })

# Individual-only alternative: mitigate every configured/inferred sensitive attribute
# separately, without intersectional pair/triple/four-way groups.
# USER_CONFIG["fairness"].update({
#     "mitigation_scope": "all_individual",
#     "mitigation_custom_attributes": None,
#     "mitigation_custom_intersections": None,
# })

# Generic CSV example:
# USER_CONFIG["data"].update({
#     "dataset_mode": "generic_tabular",
#     "dataset_name": "my_dataset",
#     "scenario": "custom",
#     "csv_path": "/content/my_dataset.csv",
#     "target_col": "outcome",
#     "positive_target_value": 1,
#     "sensitive_attrs": ("sex", "race", "ever_married", "Residence_type", "work_type"),
#     "exclude_cols": ("patient_id", "visit_date"),
#     "feature_cols": None,
# })


## 4. Configured dataset preparation and automatic feature selection

This stage applies the analyst configuration, creates the target, resolves sensitive attributes, selects predictors automatically, and creates the train/validation/test split.


In [ ]:
ctx = initialise_pipeline(USER_CONFIG)
ctx = stage_2_configured_dataset_preparation(ctx)


## 5. Model training

The notebook prints how each model has been trained, why each model/grid has been included, which parameters have been selected, and whether GPU acceleration is available. XGBoost uses a device-safe policy by default, so CPU-resident scikit-learn matrices do not trigger the CUDA/CPU DMatrix fallback warning.

In [ ]:
ctx = stage_3_train_models(ctx)


## 6. Held-out baseline evaluation and visual diagnostics

This stage displays ROC curves, confusion matrices, probability histograms, and the multi-model validation threshold-selection figure. The validation, held-out test, and fairness-gap numerical results are merged in the next stage to avoid duplicated result tables.

In [ ]:
ctx = stage_4_baseline_visual_evaluation(ctx)


## 7. Fairness audit

This stage computes subgroup FPR, FNR, TPR, selection rate and combined FPR/FNR gaps across the configured sensitive attributes. Validation metrics, held-out test metrics and baseline fairness gaps are exported in one merged model summary table.

In [ ]:
ctx = stage_5_fairness_audit(ctx)


## 8. Mitigation retraining, post-processing and statistical significance

**Corrected v18 final-run configuration:** this stage uses `bootstrap_reps=1000`, `mitigation_scope="custom_plus_pairs"`, and all requested 2-, 3-, and 4-way intersectional groups among the configured custom sensitive attributes when subgroup support is sufficient. The runtime guards are explicit and set high enough to avoid silently dropping requested groups in the included ACTG175-style run. This cell can therefore be substantially slower than the v18 smoke run.

In [ ]:
ctx = stage_6_mitigation_and_significance(ctx)


## 9. Final registry and recommendations

This stage exports a compact artifact manifest, a decision registry, and practical recommendations for improving predictive performance and mitigation analysis.

In [ ]:
ctx = stage_7_finalise(ctx)


## 10. Export all results as ZIP

This final cell compresses all generated tables and figures into a single ZIP archive. In Colab it will attempt a direct browser download; in Kaggle/Jupyter/local environments it creates a clickable file link and leaves the ZIP in the runtime output folder.


In [ ]:
ctx = stage_8_export_results_zip(ctx)
